In [24]:
import os
import sys
import subprocess
import ctypes
import sys as sys_lib

PROJECT_ROOT = r'C:\Users\doman\Downloads\Project1-main\Roo-Code'

if os.getcwd() != PROJECT_ROOT:
    os.chdir(PROJECT_ROOT)

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# 2. Import thư viện cốt lõi của Manim
from manim import *

# 3. Import toàn bộ thư viện custom của bạn

from skills.fami_lib import *
from skills.fami_assets_helper import *
from skills.fami_effects import * # Import thêm file này phòng trường hợp các hiệu ứng nằm ở đây
config.tex_template = TexTemplate()
config.tex_template.add_to_preamble(r"\usepackage[utf8]{vietnam}")
config.verbosity = "ERROR"
import warnings
warnings.filterwarnings('ignore')

In [25]:
import sys
import os
from pathlib import Path
from IPython.display import Video, display
from scipy import stats
import math
import numpy as np
from manim import *
import random

# 1. Cấu hình đường dẫn và thông số render
config.media_dir = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\media"
config.pixel_width = 720   # Độ phân giải dọc 720x1280
config.pixel_height = 1280
config.frame_rate = 15
config.disable_caching = True 
config.verbosity = "ERROR" # Tắt các dòng log thừa

# 2. Đảm bảo thư mục tồn tại
os.makedirs(config.media_dir, exist_ok=True)

# 3. Ép môi trường nhận FFmpeg (nếu cần)
ffmpeg_bin = r"C:\ffmpeg\bin"
if ffmpeg_bin not in os.environ["PATH"]:
    os.environ["PATH"] += os.pathsep + ffmpeg_bin

In [32]:
# Lưu ý: Đảm bảo FaMIBaseScene đã được định nghĩa trước đó hoặc đổi thành Scene
class mainScene(FaMIBaseScene):
    def construct(self):
        # Subscene 1
        title = self.create_title("TÍNH TOÁN SỐ BÀN THẮNG KỲ VỌNG TRONG BONG ĐÁ")
        self.play(Write(title))
        # --- Tạo khung thành ---
        goal_frame = Rectangle(width=8, height=4, color=WHITE)
        goal_label = Text("Khung thành", font_size=24).next_to(goal_frame, UP)
        
        self.play(Create(goal_frame), Write(goal_label))
        self.wait(0.5)

        # --- Tạo hệ thống chấm ---
        num_dots = 200 
        goal_ratio = 0.27
        num_goals = int(num_dots * goal_ratio)
        num_misses = num_dots - num_goals

        # Dùng VGroup để quản lý tập hợp chấm tốt hơn
        all_dots = VGroup()
        
        # Tạo chấm xanh
        for _ in range(num_goals):
            dot = Dot(point=[random.uniform(-3.8, 3.8), random.uniform(-1.8, 1.8), 0], 
                     color=GREEN, radius=0.06)
            all_dots.add(dot)
            
        # Tạo chấm đỏ
        for _ in range(num_misses):
            dot = Dot(point=[random.uniform(-3.8, 3.8), random.uniform(-1.8, 1.8), 0], 
                     color=RED, radius=0.06)
            all_dots.add(dot)

        # Xáo trộn vị trí trong VGroup để xuất hiện ngẫu nhiên
        all_dots.submobjects = random.sample(all_dots.submobjects, len(all_dots.submobjects))

        # Hiệu ứng xuất hiện
        self.play(
            LaggedStart(
                *[FadeIn(d) for d in all_dots],
                lag_ratio=0.01,
                run_time=2.5
            )
        )
        
        # Hiển thị tỷ lệ thực nghiệm
        ratio_text = MathTex(
            "P_{thuc nghiem} = \\frac{2700}{10000} = 0.27", 
            color=YELLOW
        ).scale(0.8)
        ratio_text.next_to(goal_frame, DOWN, buff=0.5)
        
        self.play(Write(ratio_text))
        self.wait(1.5)
        self.wait(1)

        # Định nghĩa Ma trận Vector X và các thành phần
        # Ma trận X nằm bên trái
        vector_x = MathTex(
            "X = \\begin{bmatrix} x_1 \\\\ x_2 \\\\ x_3 \\\\ \\vdots \\\\ x_n \\end{bmatrix}",
            color=BLUE
        ).scale(0.9).shift(LEFT * 1.5 + UP * 0.5)

        # Chú thích các biến
        labels = VGroup(
            Text("x_1: Distance", font_size=18),
            Text("x_2: Shot Angle", font_size=18),
            Text("x_3: Shot Type", font_size=18)
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.4).next_to(vector_x, RIGHT, buff=0.4)

        # ẢNH MINI
        player_img_path = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\assets\baycho.png"
        player = ImageMobject(player_img_path)
        
        player.set_width(2.5) 
        player.move_to([1.5, -2.5, 0]) # Giữ vị trí ở nửa dưới màn hình dọc
        
        player_label = Text("GOAT", font_size=20, color=YELLOW).next_to(player, DOWN, buff=0.25)
        
        # Mũi tên chĩa từ ma trận xuống hình ảnh cầu thủ mini
        arrows = VGroup(*[
            Arrow(
                start=vector_x.get_bottom(), 
                end=player.get_top(), 
                buff=0.15, # Giảm buff để mũi tên chạm sát vào ảnh nhỏ
                stroke_width=2, 
                color=GRAY_A
            ) for _ in range(2)
        ])

        # TRANSITION
        self.play(
            FadeOut(goal_frame),
            FadeOut(goal_label),
            FadeOut(ratio_text),
            # Các chấm thu nhỏ và biến mất tại vị trí ma trận
            all_dots.animate.scale(0.1).move_to(vector_x.get_center()).set_opacity(0),
            run_time=1.5,
            rate_func=slow_into
        )

        # XUẤT HIỆN VECTOR VÀ ẢNH MINI
        self.play(
            Write(vector_x),
            FadeIn(player, shift=UP), 
            Write(player_label),
            run_time=1.2
        )

        # Vẽ mũi tên và hiện chú thích đặc trưng
        self.play(
            Create(arrows),
            LaggedStart(*[Write(l) for l in labels], lag_ratio=0.5),
            run_time=1.5
        )

        self.wait(2)
        # --- FLOW MỚI: HÌNH HỌC & TRỰC QUAN HÓA XG ---
        self.wait(1)

        # STEP 1: Biến mất toàn bộ vật thể từ phân cảnh trước
        self.play(
            FadeOut(vector_x),
            FadeOut(labels),
            FadeOut(arrows),
            FadeOut(player),
            FadeOut(player_label),
            run_time=1
        )
        self.wait(0.5)

        # STEP 2: Dùng trục tọa độ để thể hiện khoảng cách phẳng
        # Khởi tạo hệ trục tọa độ nhỏ gọn phù hợp màn hình dọc
        axes = Axes(
            x_range=[0, 7, 1],
            y_range=[0, 7, 1],
            x_length=4.5,
            y_length=4.5,
            axis_config={"color": GRAY, "include_numbers": False}
        ).shift(UP * 1.5) # Đẩy hệ trục lên nửa trên màn hình

        # Định nghĩa điểm cầu thủ A và tâm khung thành B trên trục tọa độ
        pt_player = axes.c2p(2, 2, 0)
        pt_goal = axes.c2p(5, 6, 0)

        dot_a = Dot(pt_player, color=YELLOW, radius=0.08)
        lbl_a = MathTex("A(x_1, y_1)", font_size=20, color=YELLOW).next_to(dot_a, DOWN)
        
        dot_b = Dot(pt_goal, color=WHITE, radius=0.08)
        lbl_b = MathTex("B(x_2, y_2)", font_size=20, color=WHITE).next_to(dot_b, UP)

        # Đường thẳng nối hai điểm thể hiện khoảng cách d trên đồ thị
        dist_line_axes = Line(pt_player, pt_goal, color=BLUE, stroke_width=3)
        lbl_d_axes = MathTex("d", color=BLUE).scale(0.8).next_to(dist_line_axes.get_center(), UL, buff=0.1)

        self.play(Create(axes), run_time=1)
        self.play(
            FadeIn(dot_a, lbl_a),
            FadeIn(dot_b, lbl_b),
            Create(dist_line_axes),
            Write(lbl_d_axes),
            run_time=1.5
        )

        # STEP 3: Hiện công thức Euclid ngay bên cạnh (Đặt ở nửa dưới màn hình dọc)
        formula_d = MathTex(
            "d = \\sqrt{(x_2-x_1)^2 + (y_2-y_1)^2}",
            color=YELLOW
        ).scale(0.85).shift(DOWN * 2)

        self.play(Write(formula_d))
        self.wait(2.5)

        # STEP 4: Xóa hệ trục và công thức phẳng để chuyển sang thực địa góc sút
        self.play(
            FadeOut(axes),
            FadeOut(dot_a), FadeOut(lbl_a),
            FadeOut(dot_b), FadeOut(lbl_b),
            FadeOut(dist_line_axes), FadeOut(lbl_d_axes),
            FadeOut(formula_d),
            run_time=1
        )
        self.wait(0.5)

        # STEP 5: Hiện lại ảnh baycho.png (size 2.5) trước KHUNG THÀNH HOÀN CHỈNH (Cột dọc + Xà ngang)
        # Định nghĩa tọa độ các góc khung thành (Phóng to quy mô)
        goal_width = 5.0
        goal_height = 2.5
        y_ground = 1.5 # Cao độ của vạch mặt cỏ trên màn hình dọc

        post_bottom_left  = np.array([-goal_width/2, y_ground, 0])
        post_bottom_right = np.array([goal_width/2, y_ground, 0])
        goal_top_left     = np.array([-goal_width/2, y_ground + goal_height, 0])
        goal_top_right    = np.array([goal_width/2, y_ground + goal_height, 0])
        goal_center_point = np.array([0, y_ground, 0]) # Tâm vạch gôn để nối khoảng cách d

        # Tạo các thành phần cấu tạo khung thành
        left_post  = Line(post_bottom_left, goal_top_left, color=WHITE, stroke_width=4)
        right_post = Line(post_bottom_right, goal_top_right, color=WHITE, stroke_width=4)
        crossbar   = Line(goal_top_left, goal_top_right, color=WHITE, stroke_width=4)
        ground_line = Line(np.array([-3.5, y_ground, 0]), np.array([3.5, y_ground, 0]), color=GRAY_D, stroke_width=2)

        # Gom toàn bộ khung thành vào một Group để quản lý
        complete_goal = VGroup(ground_line, left_post, right_post, crossbar)
        real_goal_lbl = Text("Khung thành hoàn chỉnh", font_size=18, color=WHITE).next_to(crossbar, UP, buff=0.15)

        # Khởi tạo lại ảnh baycho.png với size 2.5 đặt ở phía dưới đối diện khung thành
        player_img_path = r"C:\Users\doman\Downloads\Project1-main\Roo-Code\assets\baycho.png"
        big_player = ImageMobject(player_img_path)
        big_player.set_width(2.5) 
        big_player.move_to([0, -2.5, 0]) # Đẩy xuống dưới một chút để kéo giãn khoảng cách với khung thành to
        
        big_player_lbl = Text("Brahim Díaz", font_size=20, color=YELLOW).next_to(big_player, DOWN, buff=0.15)

        # Hiệu ứng dựng khung thành và hiện cầu thủ
        self.play(
            Create(complete_goal),
            Write(real_goal_lbl),
            FadeIn(big_player, shift=UP),
            Write(big_player_lbl),
            run_time=1.5
        )
        self.wait(0.5)

        # STEP 6: Nối khoảng cách d (đến tâm vạch gôn) và góc sút theta (đến 2 cột dọc đáy)
        player_center = big_player.get_center()

        # Đường tính khoảng cách d thẳng tới chính giữa vạch gôn mặt đất
        real_line_d = Line(player_center, goal_center_point, color=BLUE, stroke_width=3)
        real_lbl_d = MathTex("d", color=BLUE).scale(0.9).next_to(real_line_d.get_center(), RIGHT, buff=0.15)

        # Hai đường giới hạn góc sút nối tới điểm tiếp đất của 2 cột dọc (hoặc đỉnh cột tùy ông chọn, ở đây nối xuống đáy gôn)
        ray_to_left = Line(player_center, post_bottom_left, color=GRAY_B, stroke_width=2)
        ray_to_right = Line(player_center, post_bottom_right, color=GRAY_B, stroke_width=2)

        # Tính toán vẽ vòng cung góc sút Theta xuất phát từ góc giữa 2 đường ray
        theta_arc = ArcBetweenPoints(
            start=player_center + 0.5 * normalize(post_bottom_left - player_center),
            end=player_center + 0.5 * normalize(post_bottom_right - player_center),
            stroke_width=2.5,
            color=GREEN
        )
        real_lbl_theta = MathTex("\\theta", color=GREEN).scale(0.9).next_to(theta_arc, UP, buff=0.15)

        # Thực hiện hiệu ứng vẽ đồ họa thực địa rộng rãi
        self.play(Create(real_line_d), Write(real_lbl_d), run_time=1)
        self.play(
            Create(ray_to_left),
            Create(ray_to_right),
            run_time=1
        )
        self.play(
            Create(theta_arc),
            Write(real_lbl_theta),
            run_time=0.8
        )
        self.wait(2)
######################
scene = mainScene()
scene.render()
video_file = Path(config.media_dir) / "videos" / "1280p15" / "mainScene.mp4"
if video_file.exists():
    display(Video(str(video_file), embed=True, width=300))
else:
    found = list(Path(config.media_dir).glob("**/mainScene.mp4"))
    if found:
        display(Video(str(found[0]), embed=True, width=300))
    else:
        print("Render xong nhưng không tìm thấy file mp4 để hiển thị.")